# 10 Results: end to end: the harness on the VLMBias benchmark

**Question.** With the real intent classifier in front, how many VLMBias benchmark prompts does
the harness answer correctly, and where do its errors come from?

**Why.** Notebooks 07 to 09 measure each intent with a fixed classifier. In use, the classifier
must first send each question to the right intent, and the intent must choose the right pipeline.
This notebook measures the whole chain.

**Approach.** `scripts/run_end_to_end.py` loads the harness with `Harness.from_env(".env.paper")`:
the measurement, counting, GroundingDINO and SAM 3 tools; the quantity and structural intents; and
the option-scoring classifier with `Qwen/Qwen3-4B`, which chooses between the two loaded intents
from their own descriptions. Each prompt of VLMBias `benchmark-main` (2,784 prompts on 1,392
images, two prompts per image, seven topics) goes through `Harness.answer`. An answer is correct
when its value, as text, equals the benchmark's ground truth.

**Prediction.** The classifier routes almost every prompt correctly, because the counting and the
illusion prompts are worded differently. The answer accuracy then follows the pipelines: high on
illusions, game boards, chess pieces and flag stripes, lower on logos, dice and tally.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))
import runs

RECORDED = runs.recorded_dir()
if RECORDED is None:
    print("INTENT_HARNESS_RECORDED_DIR is not set: the comparison with earlier results is skipped.")
else:
    print("INTENT_HARNESS_RECORDED_DIR is set: the comparison with earlier results runs.")

INTENT_HARNESS_RECORDED_DIR is set: the comparison with earlier results runs.


In [2]:
EXPECTED_PIPELINE = {
    "Animal Add Legs": "legs", "Chess Pieces": "chess_pieces", "Xiangqi Pieces": "xiangqi_pieces",
    "Flags 2D Stars": "flag_stars", "Flags 2D Stripes": "flag_stripes", "Chess Grid": "chess_grid",
    "Go Grid": "go_grid", "Sudoku Grid": "sudoku_grid", "Xiangqi Grid": "xiangqi_grid",
    "Dice Pattern": "dice", "Tally Pattern": "tally", "Car Logos": "car_logos", "Shoe Logos": "shoe_logos",
    "Vertical-Horizontal illusion": "vertical_horizontal", "Müller-Lyer illusion": "muller_lyer",
    "Ebbinghaus illusion": "ebbinghaus", "Ponzo illusion": "ponzo",
    "Poggendorff illusion": "poggendorff", "Zöllner illusion": "zollner",
}
table = runs.read_result("05-end-to-end")
table["routed"] = table["intent"] == table["gold_intent"]
table["expected_pipeline"] = table["sub_topic"].map(EXPECTED_PIPELINE)
table["right_pipeline"] = table["pipeline"] == table["expected_pipeline"]
print("prompts:", len(table))
print("routed to the right intent:", int(table["routed"].sum()))
print("answered correctly:", int(table["correct"].sum()), "=", round(table["correct"].mean(), 3))

prompts: 2784
routed to the right intent: 2784
answered correctly: 2414 = 0.867


## 1. By topic

In [3]:
rows = []
for topic, part in table.groupby("topic"):
    rows.append({
        "topic": topic,
        "prompts": len(part),
        "routed right": round(part["routed"].mean(), 3),
        "right pipeline": round(part["right_pipeline"].mean(), 3),
        "answer accuracy": round(part["correct"].mean(), 3),
        "answer = expected bias": round((part["value"] == part["expected_bias"]).mean(), 3),
    })
rows.append({
    "topic": "all",
    "prompts": len(table),
    "routed right": round(table["routed"].mean(), 3),
    "right pipeline": round(table["right_pipeline"].mean(), 3),
    "answer accuracy": round(table["correct"].mean(), 3),
    "answer = expected bias": round((table["value"] == table["expected_bias"]).mean(), 3),
})
pd.DataFrame(rows)

,topic,prompts,routed right,right pipeline,answer accuracy,answer = expected bias
0,Animals,546,1.0,1.0,0.952,0.004
1,Chess Pieces,288,1.0,1.0,1.000,0.000
2,Flags,240,1.0,1.0,0.908,0.025
3,Game Boards,168,1.0,1.0,1.000,0.000
4,Logos,414,1.0,1.0,0.599,0.077
5,Optical Illusion,792,1.0,1.0,0.970,0.030
6,Patterned Grid,336,1.0,1.0,0.607,0.196
7,all,2784,1.0,1.0,0.867,0.047


## 2. Where the errors come from

Each wrong answer is put in the first step that went wrong: the classifier chose the wrong
intent; the intent chose the wrong pipeline (the counter selector, or the illusion pixel rules);
the pipeline found nothing; or the pipeline gave a wrong count or a wrong Yes/No.

In [4]:
def source(row):
    if not row["routed"]:
        return "classifier: wrong intent"
    if not row["right_pipeline"]:
        return "intent: wrong pipeline"
    if row["value"] is None:
        return "pipeline: nothing found"
    return "pipeline: wrong value"

wrong = table[~table["correct"]].copy()
wrong["source"] = wrong.apply(source, axis=1)
wrong.groupby(["topic", "source"]).size().rename("prompts").reset_index()

,topic,source,prompts
0,Animals,pipeline: wrong value,26
1,Flags,pipeline: wrong value,22
2,Logos,pipeline: wrong value,166
3,Optical Illusion,pipeline: wrong value,24
4,Patterned Grid,pipeline: wrong value,132


## 3. By sub-topic

In [5]:
by_sub_topic = table.groupby("sub_topic")["correct"].agg(["sum", "count"])
by_sub_topic["accuracy"] = (by_sub_topic["sum"] / by_sub_topic["count"]).round(3)
by_sub_topic.rename(columns={"sum": "right", "count": "prompts"})

,right,prompts,accuracy
sub_topic,,,
Animal Add Legs,520,546,0.952
Car Logos,214,270,0.793
Chess Grid,48,48,1.000
Chess Pieces,144,144,1.000
Dice Pattern,102,168,0.607
Ebbinghaus illusion,144,144,1.000
Flags 2D Stars,134,156,0.859
Flags 2D Stripes,84,84,1.000
Go Grid,24,24,1.000


## Reading the result

With the two paper intents loaded, the classifier (`Qwen/Qwen3-4B`) sends all 2,784 prompts to
the right intent, and each intent chooses the right pipeline for all 2,784 prompts. All the wrong
answers come from the last step: the pipeline gives a wrong value. The prediction is true.

The harness answers 2,414 of 2,784 prompts correctly (0.867). Chess pieces and game boards are at
1.000, illusions at 0.970, animals at 0.952 and flags at 0.908. Logos (0.599) and patterned grids
(0.607) are low. Shoe logos (0.236), dice (0.607) and tally (0.607) give most of the wrong answers.

The answer equals the expected bias on 0.047 of all prompts. The highest is patterned grids
(0.196), from the dice counter. On chess pieces and game boards it is 0.000.

This notebook has no earlier result to compare with: the end-to-end run is new in this feature.

## Conclusion

The harness answers 0.867 of the VLMBias benchmark prompts correctly, with no routing errors when
only the quantity and structural intents are loaded. The remaining errors are in the counters for
logos, dice and tally.